In [1]:
import pandas as pd
import numpy as np
import os

folder = r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\CIC-IDS2017"

# List all CSV files in the folder
files = [f for f in os.listdir(folder) if f.endswith('.csv') or 'pcap_ISCX' in f]
print("Files found:", files)

Files found: ['Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', 'Friday-WorkingHours-Morning.pcap_ISCX.csv', 'Monday-WorkingHours.pcap_ISCX.csv', 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', 'Tuesday-WorkingHours.pcap_ISCX.csv', 'Wednesday-workingHours.pcap_ISCX.csv']


In [2]:
dataframes = []

for file in files:
    filepath = os.path.join(folder, file)
    df = pd.read_csv(filepath, low_memory=False)
    
    # Strip whitespace from column names (a well-documented CICIDS2017 quirk)
    df.columns = df.columns.str.strip()
    
    dataframes.append(df)
    print(f"{file}: {df.shape}")

# Combine all 8 days into a single dataframe
cicids_df = pd.concat(dataframes, ignore_index=True)

print("\nCombined shape:", cicids_df.shape)

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: (225745, 79)
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: (286467, 79)
Friday-WorkingHours-Morning.pcap_ISCX.csv: (191033, 79)
Monday-WorkingHours.pcap_ISCX.csv: (529918, 79)
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: (288602, 79)
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: (170366, 79)
Tuesday-WorkingHours.pcap_ISCX.csv: (445909, 79)
Wednesday-workingHours.pcap_ISCX.csv: (692703, 79)

Combined shape: (2830743, 79)


In [3]:
print(cicids_df['Label'].value_counts())

Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [4]:
# Fix the broken encoding in Web Attack labels
cicids_df['Label'] = cicids_df['Label'].str.replace('�', '-', regex=False)

print(cicids_df['Label'].value_counts())

Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack - Brute Force         1507
Web Attack - XSS                  652
Infiltration                       36
Web Attack - Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [5]:
# Check for missing values
print("Missing values per column (top 10):")
print(cicids_df.isnull().sum().sort_values(ascending=False).head(10))

# Check for infinite values (common issue in CICIDS2017's Flow Bytes/s and Flow Packets/s columns)
numeric_cols = cicids_df.select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(cicids_df[numeric_cols]).sum()
print("\nColumns with infinite values:")
print(inf_counts[inf_counts > 0])

Missing values per column (top 10):
Flow Bytes/s            1358
Destination Port           0
Bwd Avg Bytes/Bulk         0
Fwd Avg Packets/Bulk       0
Fwd Avg Bytes/Bulk         0
Fwd Header Length.1        0
Avg Bwd Segment Size       0
Avg Fwd Segment Size       0
Average Packet Size        0
Down/Up Ratio              0
dtype: int64

Columns with infinite values:
Flow Bytes/s      1509
Flow Packets/s    2867
dtype: int64


In [6]:
before_rows = cicids_df.shape[0]

# Replace infinite values with NaN, then drop any row with NaN in these columns
cicids_df.replace([np.inf, -np.inf], np.nan, inplace=True)
cicids_df.dropna(inplace=True)

after_rows = cicids_df.shape[0]

print(f"Removed {before_rows - after_rows} rows with missing/infinite values")
print("New shape:", cicids_df.shape)

Removed 2867 rows with missing/infinite values
New shape: (2827876, 79)


In [7]:
# Check duplicates
print("Duplicate rows:", cicids_df.duplicated().sum())

# Check data types - see if any columns are text/object type (need encoding)
print("\nColumn data types:")
print(cicids_df.dtypes.value_counts())

print("\nNon-numeric columns:")
print(cicids_df.select_dtypes(include='object').columns.tolist())

Duplicate rows: 307078

Column data types:
int64      54
float64    24
object      1
Name: count, dtype: int64

Non-numeric columns:
['Label']


In [8]:
before_rows = cicids_df.shape[0]

cicids_df = cicids_df.drop_duplicates()

after_rows = cicids_df.shape[0]
print(f"Removed {before_rows - after_rows} duplicate rows")
print("New shape:", cicids_df.shape)

Removed 307078 duplicate rows
New shape: (2520798, 79)


In [9]:
cicids_df['binary_label'] = cicids_df['Label'].apply(lambda x: 'BENIGN' if x == 'BENIGN' else 'ATTACK')

print(cicids_df['binary_label'].value_counts())

binary_label
BENIGN    2095057
ATTACK     425741
Name: count, dtype: int64


In [10]:
from sklearn.preprocessing import StandardScaler

feature_cols = [col for col in cicids_df.columns if col not in ['Label', 'binary_label']]

scaler = StandardScaler()
cicids_df[feature_cols] = scaler.fit_transform(cicids_df[feature_cols])

print("Features scaled successfully!")
cicids_df[feature_cols].head()

Features scaled successfully!


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
0,2.428596,-0.470914,-0.010425,-0.010950,-0.056662,-0.007566,-0.297774,-0.217169,-0.294064,-0.260452,...,-0.007428,0.002698,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764
1,2.438537,-0.470911,-0.011684,-0.010003,-0.057228,-0.007563,-0.297774,-0.217169,-0.294064,-0.260452,...,-0.008911,0.002698,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764
2,2.438590,-0.470913,-0.011684,-0.010003,-0.057228,-0.007563,-0.297774,-0.217169,-0.294064,-0.260452,...,-0.008911,0.002698,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764
3,1.974744,-0.470913,-0.011684,-0.010003,-0.057228,-0.007563,-0.297774,-0.217169,-0.294064,-0.260452,...,-0.008911,0.002698,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764
4,2.428491,-0.470914,-0.010425,-0.010950,-0.056662,-0.007566,-0.297774,-0.217169,-0.294064,-0.260452,...,-0.007428,0.002698,-0.13337,-0.110882,-0.158458,-0.107114,-0.375777,-0.11608,-0.381137,-0.361764


In [11]:
from sklearn.model_selection import train_test_split

X = cicids_df[feature_cols]
y_binary = cicids_df['binary_label']
y_multiclass = cicids_df['Label']

# Stratified split preserves class proportions in both sets
train_idx, test_idx = train_test_split(
    cicids_df.index, test_size=0.2, stratify=y_binary, random_state=42
)

train_df = cicids_df.loc[train_idx]
test_df = cicids_df.loc[test_idx]

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("\nTrain binary label balance:")
print(train_df['binary_label'].value_counts())

Train shape: (2016638, 80)
Test shape: (504160, 80)

Train binary label balance:
binary_label
BENIGN    1676045
ATTACK     340593
Name: count, dtype: int64


In [12]:
import os

processed_folder = r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\CIC-IDS2017\processed"
os.makedirs(processed_folder, exist_ok=True)

train_save_path = os.path.join(processed_folder, "CICIDS2017_train_cleaned.csv")
test_save_path  = os.path.join(processed_folder, "CICIDS2017_test_cleaned.csv")

train_df.to_csv(train_save_path, index=False)
test_df.to_csv(test_save_path, index=False)

print("Cleaned data saved successfully!")
print("Train saved to:", train_save_path)
print("Test saved to:", test_save_path)

Cleaned data saved successfully!
Train saved to: C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\CIC-IDS2017\processed\CICIDS2017_train_cleaned.csv
Test saved to: C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\CIC-IDS2017\processed\CICIDS2017_test_cleaned.csv
